In [1]:
import numpy as np
from numba import cuda

@cuda.jit
def cumulative_sum_kernel(inp, out):
    idx = cuda.grid(1)

    if idx < len(inp):
        s = 0

        for i in range(idx + 1):
            s += inp[i]

        out[idx] = s

# Sample data
input_array = np.array([1, 2, 3, 4, 5], dtype=np.int32)

output_array = np.zeros_like(input_array)

# Copy to GPU
d_input = cuda.to_device(input_array)
d_output = cuda.device_array_like(output_array)

threads_per_block = 256
blocks_per_grid = (len(input_array) + threads_per_block - 1) // threads_per_block

# Launch kernel
cumulative_sum_kernel[blocks_per_grid, threads_per_block](d_input, d_output)

# Copy result back
output_array = d_output.copy_to_host()

print("Input :", input_array)
print("Output:", output_array)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Input : [1 2 3 4 5]
Output: [ 1  3  6 10 15]
